In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

os.getcwd()

'C:\\Users\\abhin\\OneDrive\\Desktop\\Data_Science\\Research_Projects\\Uplifting_News'

In [2]:
import transformers
import sentence_transformers
import datasets
import setfit
import huggingface_hub

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("datasets:", datasets.__version__)
print("setfit:", setfit.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

transformers: 4.57.6
sentence-transformers: 2.6.1
datasets: 2.16.1
setfit: 0.7.0
huggingface_hub: 0.36.2


## 1. Data Sourcing

Let's create url and headline data for uplifitng rss feeds.

In [ ]:
# ### Code to extract non-reddit links from good news org
# import feedparser
# import pandas as pd
# from bs4 import BeautifulSoup
# from urllib.parse import urlparse

# rss_url = "https://www.goodnewsnetwork.org/category/news/feed/?limit=100"

# def is_reddit_url(url: str) -> bool:
#     try:
#         netloc = urlparse(url).netloc.lower()
#         return "reddit.com" in netloc or "redd.it" in netloc
#     except:
#         return False

# def get_actual_url(entry):
#     if hasattr(entry, "links"):
#         for link_obj in entry.links:
#             href = link_obj.get("href")
#             if href and not is_reddit_url(href):
#                 return href

#     if hasattr(entry, "content"):
#         for block in entry.content:
#             html = block.get("value", "")
#             soup = BeautifulSoup(html, "html.parser")
#             for a in soup.find_all("a", href=True):
#                 href = a["href"]
#                 if not is_reddit_url(href):
#                     return href

#     if hasattr(entry, "summary"):
#         soup = BeautifulSoup(entry.summary, "html.parser")
#         for a in soup.find_all("a", href=True):
#             href = a["href"]
#             if not is_reddit_url(href):
#                 return href

#     return entry.link

# feed = feedparser.parse(
#     rss_url
# )

# rows = []
# for entry in feed.entries:
#     rows.append({
#         "url": get_actual_url(entry),
#         "headline": entry.title.strip()
#     })

# df = pd.DataFrame(rows)

# # Keep only rows where URL is NOT Reddit
# df = df[~df["url"].str.contains(r"reddit\.com|redd\.it", case=False, na=False)].copy()

# # Optional: remove duplicates
# df = df.drop_duplicates(subset=["url", "headline"]).reset_index(drop=True)

# df.to_csv("Good_News_external.csv", index=False)

# print(df.head())
# print(f"Saved {len(df)} non-Reddit rows to goodnews_external_only.csv")

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://apnews.com/hub/apf-topnews"

headers = {
    "User-Agent": "Mozilla/5.0"
}

resp = requests.get(url, headers=headers, timeout=20)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

rows = []
seen = set()

for a in soup.find_all("a", href=True):
    href = a["href"]
    text = a.get_text(" ", strip=True)

    if not text:
        continue

    full_url = urljoin("https://apnews.com", href)

    # keep AP article-like links only
    if "apnews.com/article/" not in full_url:
        continue

    key = (full_url, text)
    if key in seen:
        continue
    seen.add(key)

    rows.append({
        "url": full_url,
        "headline": text
    })

df = pd.DataFrame(rows).drop_duplicates(subset=["url", "headline"]).reset_index(drop=True)

df.to_csv("ap_topnews_headlines.csv", index=False)

print(df.head(20))
print(f"Saved {len(df)} rows to ap_topnews_headlines.csv")

In [ ]:
### CBS News Main

import feedparser
import pandas as pd

rss_url = "https://www.cbsnews.com/latest/rss/main?limit=75"

feed = feedparser.parse(
    rss_url,
    request_headers={"User-Agent": "Mozilla/5.0"}
)

rows = []
for entry in feed.entries:
    headline = getattr(entry, "title", "").strip()
    url = getattr(entry, "link", "").strip()

    if headline and url:
        rows.append({
            "url": url,
            "headline": headline
        })

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["url", "headline"]).reset_index(drop=True)

df.to_csv("cbs_latest_headlines.csv", index=False)

print(df.head())
print(f"Saved {len(df)} unique rows to cbs_latest_headlines.csv")

In [ ]:
### BBC News Feed

rss_url = "https://feeds.bbci.co.uk/news/world/rss.xml?limit=75"

feed = feedparser.parse(
    rss_url,
    request_headers={"User-Agent": "Mozilla/5.0"}
)

rows = []
for entry in feed.entries:
    headline = getattr(entry, "title", "").strip()
    url = getattr(entry, "link", "").strip()

    if headline and url:
        rows.append({
            "url": url,
            "headline": headline
        })

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["url", "headline"]).reset_index(drop=True)

df.to_csv("bbc_latest_headlines.csv", index=False)

print(df.head())
print(f"Saved {len(df)} unique rows to bbc_latest_headlines.csv")

## Section-2: Model Training

In [ ]:
import pandas as pd
import re

df = pd.read_excel("news_sources.xlsx", sheet_name="Uplifting_Labeled_Data")

df = df[["headline", "label"]].dropna()

def clean_headline(text):
    text = str(text)

    # remove urls
    text = re.sub(r"http\S+", "", text)

    # remove html entities
    text = re.sub(r"&\w+;", " ", text)

    # keep letters/numbers/basic punctuation
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'-]", " ", text)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip().lower()

df["clean_headline"] = df["headline"].apply(clean_headline)

print(df.shape)
print(df.head())

In [ ]:
## Convert labels to binary
df["label_binary"] = df["label"].map({
    "Uplifting": 1,
    "Not Uplifting": 0
})

print(df['label_binary'].value_counts())
df.head(5)

In [ ]:
## train-val-test + TF-IDF
from sklearn.model_selection import train_test_split

X = df["clean_headline"]
y = df["label_binary"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
################ Baseline model-1: Logistic Regression ##################

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=2000)
model.fit(X_train_vec, y_train)
val_preds = model.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))

## Test accuracy
test_preds_lr = model.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, test_preds_lr))
print(classification_report(y_test, test_preds_lr))

In [ ]:
############### Baseline model-2: Random Forest #############

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train_vec, y_train)
preds = rf.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, preds))

## Test accuracy
test_preds_rf = rf.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, test_preds_rf))
print(classification_report(y_test, test_preds_rf))

In [ ]:
########### Baseline model-3: SVM #############
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_vec, y_train)
val_preds = svm.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))

## Test accuracy
test_preds_svm = svm.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, test_preds_svm))
print(classification_report(y_test, test_preds_svm))

In [ ]:
##### Baseline model-4: Naive Bayes ##########

from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
preds = nb.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, preds))

## Test accuracy
test_preds_nb = nb.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, test_preds_nb))
print(classification_report(y_test, test_preds_nb))

In [ ]:
########### Enhanaced model-1: Sentence Transofmers + Logistic Regression #############
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer

## Load data
df = pd.read_excel("news_sources.xlsx", sheet_name="Uplifting_Labeled_Data")
df = df[["headline", "label"]].dropna()

## Clean it
def clean_text(text):
    text = str(text)
    
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()
df["clean_headline"] = df["headline"].apply(clean_text)

## Convert labels to binary
df["label_binary"] = df["label"].map({
    "Uplifting": 1,
    "Not Uplifting": 0
})

## Train-val-test split
X = df["clean_headline"]
y = df["label_binary"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

## Convert clean headlines to embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = embedding_model.encode(X_train.tolist(), show_progress_bar=True, normalize_embeddings=True)
X_val_emb = embedding_model.encode(X_val.tolist(), normalize_embeddings=True)
X_test_emb = embedding_model.encode(X_test.tolist(), normalize_embeddings=True)

################ Sentence Embedding + Logistic Regression ################
model = LogisticRegression(max_iter=2000)
model.fit(X_train_emb, y_train)
val_preds = model.predict(X_val_emb)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))

# Test accuracy
test_preds_lr = model.predict(X_test_emb)
print("Test Accuracy:", accuracy_score(y_test, test_preds_lr))
print(classification_report(y_test, test_preds_lr))

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

scores = cross_val_score(
    LogisticRegression(max_iter=2000),
    X_train_emb,
    y_train,
    cv=5
)

print("CV accuracy:", scores.mean())

In [ ]:
def predict_headline(text):    
    text = clean_text(text)    
    emb = embedding_model.encode([text], normalize_embeddings=True)    
    pred = model.predict(emb)[0]    
    return "Uplifting" if pred == 1 else "Not Uplifting"

predict_headline("Community raises funds to rebuild local library")

In [ ]:
## Testing 4 more examples
predict_headline("Community raises funds to rebuild local library")
print("Example-2: ", predict_headline("Scientists warn global temperatures may exceed targets despite clean energy progress"))
print("Example-3: ", predict_headline("New cancer drug shows promise but researchers say more trials are needed"))
print("Example-4: ", predict_headline("City approves ambitious climate plan after years of rising emissions"))
print("Example-5: ", predict_headline("Wildlife populations stabilize in some regions despite global biodiversity decline"))

In [ ]:
### One more iteration
############################################################
# UPLIFTING NEWS CLASSIFIER
# SentenceTransformer + Logistic Regression
############################################################

import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_score

from sentence_transformers import SentenceTransformer


############################################################
# 1. LOAD DATA
############################################################
df = pd.read_excel("news_sources.xlsx", sheet_name="Uplifting_Labeled_Data")
df = df[["headline", "label"]].dropna()

############################################################
# 2. CLEAN TEXT
############################################################
def clean_text(text):

    text = str(text)

    text = re.sub(r"http\S+", "", text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_headline"] = df["headline"].apply(clean_text)

############################################################
# 3. CONVERT LABELS
############################################################
df["label_binary"] = df["label"].map({
    "Uplifting": 1,
    "Not Uplifting": 0
})


############################################################
# 4. TRAIN / VALIDATION / TEST SPLIT
############################################################
X = df["clean_headline"]
y = df["label_binary"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

############################################################
# 5. LOAD SENTENCE TRANSFORMER
############################################################
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

############################################################
# 6. CREATE EMBEDDINGS
############################################################
X_train_emb = embedding_model.encode(X_train.tolist(), show_progress_bar=True, normalize_embeddings=True)
X_val_emb = embedding_model.encode(X_val.tolist(), normalize_embeddings=True)
X_test_emb = embedding_model.encode(X_test.tolist(), normalize_embeddings=True)

############################################################
# 7. TRAIN LOGISTIC REGRESSION
############################################################
model = LogisticRegression(max_iter=2000)
model.fit(X_train_emb, y_train)

############################################################
# 8. VALIDATION PERFORMANCE
############################################################
val_preds = model.predict(X_val_emb)
print("\nValidation Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

############################################################
# 9. TEST PERFORMANCE
############################################################
test_preds = model.predict(X_test_emb)
print("\nTest Accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))


############################################################
# 10. PROBABILITY OUTPUT
############################################################
val_probs = model.predict_proba(X_val_emb)[:,1]
test_probs = model.predict_proba(X_test_emb)[:,1]

############################################################
# 11. FIND PRECISION THRESHOLD
############################################################
print("\nThreshold tuning for high precision:\n")

for t in np.arange(0.50, 0.95, 0.05):
    preds = (val_probs > t).astype(int)
    precision = precision_score(y_val, preds)
    print("threshold:", round(t,2), "precision:", round(precision,2))


#### Sweet spot seems to be threshold=0.80, without being too strict!

In [ ]:
############################################################
# 12. APPLY CHOSEN THRESHOLD
############################################################
threshold = 0.80
test_preds_thresh = (test_probs > threshold).astype(int)

print("\nTest Accuracy with threshold:", accuracy_score(y_test, test_preds_thresh))
print(classification_report(y_test, test_preds_thresh))

############################################################
# 13. PREDICTION FUNCTION FOR APP
############################################################
def predict_headline(text, threshold=0.75):

    text = clean_text(text)
    emb = embedding_model.encode([text], normalize_embeddings=True)
    prob = model.predict_proba(emb)[0][1]

    if prob > threshold:
        label = "Uplifting"
    else:
        label = "Not Uplifting"

    return label, prob

############################################################
# 14. TEST EXAMPLES
############################################################

print("\nExample predictions:\n")
print("Example-1: ", predict_headline("Community raises funds to rebuild local library"))
print("Example-2: ", predict_headline("Scientists warn global temperatures may exceed targets"))
print("Example-3: ", predict_headline("New cancer drug shows promise but researchers say more trials are needed"))
print("Example-4: ", predict_headline("City approves ambitious climate plan after years of rising emissions"))
print("Example-5: ", predict_headline("Wildlife populations stabilize in some regions despite global biodiversity decline"))


## SetFit Enhancement

We now use a method called SetFit, that takes the embeddings of the Uplifting and Not Uplifting examples, and then alters the embeddings so that postive and negative classes cluster better than a regular semantic embedding. 

In [3]:
############################################################
# UPLIFTING NEWS CLASSIFIER (SETFIT VERSION)
############################################################

import pandas as pd
import numpy as np
import re
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from datasets import Dataset
from setfit import SetFitModel, SetFitTrainer


############################################################
# LOAD DATA
############################################################

df = pd.read_excel("news_sources.xlsx", sheet_name="Uplifting_Labeled_Data")
df = df[["headline", "label"]].dropna()

############################################################
# CLEAN TEXT
############################################################

def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_headline"] = df["headline"].apply(clean_text)

############################################################
# LABEL ENCODING
############################################################

df["label_binary"] = df["label"].map({
    "Uplifting": 1,
    "Not Uplifting": 0
})

print(df["label_binary"].value_counts())
############################################################
# TRAIN / VALIDATION / TEST SPLIT
############################################################

X = df["clean_headline"]
y = df["label_binary"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

############################################################
# CONVERT TO HUGGINGFACE DATASETS
############################################################

train_dataset = Dataset.from_dict({
    "text": list(X_train),
    "label": list(y_train)
})

val_dataset = Dataset.from_dict({
    "text": list(X_val),
    "label": list(y_val)
})

test_dataset = Dataset.from_dict({
    "text": list(X_test),
    "label": list(y_test)
})

############################################################
# LOAD SETFIT MODEL
############################################################

model = SetFitModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

############################################################
# TRAINER
############################################################

trainer = SetFitTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    batch_size=16,
    num_iterations=20,     # pair generation iterations
    num_epochs=1           # classifier training epochs
)

############################################################
# TRAIN MODEL
############################################################

trainer.train()

############################################################
# VALIDATION PERFORMANCE
############################################################
val_preds = trainer.model.predict(list(X_val))
print("\nValidation Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

############################################################
# TEST PERFORMANCE
############################################################

test_preds = trainer.model.predict(list(X_test))
print("\nTest Accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))

############################################################
# PREDICTION FUNCTION
############################################################

def predict_headline(text):
    text = clean_text(text)
    pred = trainer.model.predict([text])[0]

    if pred == 1:
        label = "Uplifting"
    else:
        label = "Not Uplifting"

    return label

############################################################
# TEST EXAMPLES
############################################################

print("\nExample predictions:\n")

print("Example-1:", predict_headline("Community raises funds to rebuild local library"))
print("Example-2:", predict_headline("Scientists warn global temperatures may exceed targets despite clean energy progress"))
print("Example-3:", predict_headline("New cancer drug shows promise but researchers say more trials are needed"))
print("Example-4:", predict_headline("City approves ambitious climate plan after years of rising emissions"))
print("Example-5:", predict_headline("Wildlife populations stabilize in some regions despite global biodiversity decline"))

############################################################
# SAVE MODEL
############################################################

trainer.model.save_pretrained("uplifting_news_classifier")

print("\nSetFit model saved successfully.")

label_binary
1    278
0    213
Name: count, dtype: int64


`SentenceTransformer._target_device` has been removed, please use `SentenceTransformer.device` instead.
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Generating Training Pairs:   0%|          | 0/20 [00:00<?, ?it/s]

***** Running training *****
  Num examples = 13720
  Num epochs = 1
  Total optimization steps = 858
  Total train batch size = 16


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/858 [00:00<?, ?it/s]


Validation Accuracy: 0.8108108108108109
              precision    recall  f1-score   support

           0       0.82      0.72      0.77        32
           1       0.80      0.88      0.84        42

    accuracy                           0.81        74
   macro avg       0.81      0.80      0.80        74
weighted avg       0.81      0.81      0.81        74


Test Accuracy: 0.9594594594594594
              precision    recall  f1-score   support

           0       1.00      0.91      0.95        32
           1       0.93      1.00      0.97        42

    accuracy                           0.96        74
   macro avg       0.97      0.95      0.96        74
weighted avg       0.96      0.96      0.96        74


Example predictions:

Example-1: Uplifting
Example-2: Uplifting
Example-3: Not Uplifting
Example-4: Uplifting
Example-5: Uplifting

SetFit model saved successfully.


In [4]:
print("Example-7:", predict_headline("Renewable Energy and National Security"))
print("Example-8:", predict_headline("Europe invested â‚¬45bn in new wind energy in 2025, market tampering would put future investments at acute risk"))
print("Example-9:", predict_headline("Trump EPA repeals Endangerment Finding, stripping legal bedrock for solar and clean energy"))

Example-7: Not Uplifting
Example-8: Not Uplifting
Example-9: Not Uplifting


In [5]:
print("Example-10:", predict_headline("US destroys mine-laying vessels as Trump warns Iran over Strait of Hormuz"))
print("Example-11:", predict_headline("Australia granst asylum to two more Iranina women soccer players"))
print("Example-12:", predict_headline("Chile declared leprosy-free"))

Example-10: Not Uplifting
Example-11: Uplifting
Example-12: Uplifting


In [8]:
### Load model and use it for inference

from setfit import SetFitModel

model_loaded = SetFitModel.from_pretrained("uplifting_news_classifier")

def predict_headline(text, model):
    text = clean_text(text)
    pred = model.predict([text])[0]

    if pred == 1:
        return "Uplifting"
    else:
        return "Not Uplifting"

print("Example-L1:", predict_headline("US destroys mine-laying vessels as Trump warns Iran over Strait of Hormuz", model_loaded))
print("Example-L2:", predict_headline("Australia granst asylum to two more Iranina women soccer players", model_loaded))
print("Example-L3:", predict_headline("Chile declared leprosy-free", model_loaded))

`SentenceTransformer._target_device` has been removed, please use `SentenceTransformer.device` instead.


Example-L1: Not Uplifting
Example-L2: Uplifting
Example-L3: Uplifting


In [12]:
predict_headline("Sources: U.S. military lab tested device that may be tied to Havana Syndrome", model_loaded)

'Uplifting'

In [10]:
predict_headline("Rooftop AgriVoltaics: Lowered water usage, reduced temperatures and energy costs, increased local food security, more clean energy with evapotranspiration cooling the panels, climate-resilient public green spaces, plus leafy greens, herbs, culinary saffron, and other crops growing better", model_loaded)

'Uplifting'

In [13]:
predict_headline("Huge satellite crash down to earth", model_loaded)

'Uplifting'

In [16]:
import pandas as pd

outputs_path = 'C:\\Users\\abhin\\OneDrive\\Desktop\\Data_Science\\Research_Projects\\Uplifting_News\\outputs'

df = pd.read_csv(os.path.join(outputs_path, 'uplifting_news_2026-03-19_14-13-27.csv'))
df.head(3)

,title,rss_link,original_url,published,description,reddit_post_body,external_article_body,article_body,source,uplifting_score,primary_location,detected_locations,us_state,country_region,story_date,story_date_source
0,The King Charles III England Coast Path was op...,https://www.reddit.com/r/UpliftingNews/comment...,https://www.bbc.co.uk/news/articles/cy0dxexdd8xo,2026-03-19 14:21:24,submitted by /u/whatatwit [link] [comments],Reddit - The heart of the internet,"2,689 miles and 18 years in the making, King C...","2,689 miles and 18 years in the making, King C...",UpliftingNews,0.982411,England,"England, Scotland, Wales",NaN,England,2026-03-19,relative_today
1,We've crossed the threshold. Solar and Wind ar...,https://www.reddit.com/r/OptimistsUnite/commen...,https://ourworldindata.org/grapher/solar-pv-pr...,2026-03-17 17:18:36,Solar and Wind are the cheapest forms of energ...,Solar and Wind are the cheapest forms of energ...,What you should know about this indicator\n\nS...,Solar and Wind are the cheapest forms of energ...,OptimistsUnite,0.982248,USA,USA,NaN,USA,2026-03-17,published
2,Park Where Jackie Robinson First Played Baseba...,https://www.goodnewsnetwork.org/park-where-jac...,https://www.goodnewsnetwork.org/park-where-jac...,2026-03-13 11:00:39,The first baseball park that Jackie Robinson e...,NaN,Good News Horoscopes USA World Inspiring Anima...,Good News Horoscopes USA World Inspiring Anima...,GoodNewsNetwork,0.981857,Florida,"Florida, USA, Australia",Florida,USA,2026-03-13,published


In [2]:
# for index, row in df.head(8).iterrows():
#     #print(row['title'], '\n', row['article_body'])
#     print('##'*50)

## News Summarizer

In [13]:
article = """
After years of fierce resistance from conservationists, scientists, local residents, and lawmakers, thousands of acres of land near the Okefenokee National Wildlife Refuge will now be protected instead of turned into a titanium mine.

The state of Georgia plans to acquire around 4,000 acres near the refuge from The Conservation Fund, the group that last year purchased nearly 8,000 acres of land and mineral rights from Twin Pines Minerals in order to stop the long-contested mining proposal.

For years, the land along Trail Ridge, near the southeastern edge of the Okefenokee, had become the site of a major environmental battle. Twin Pines had pushed for approval to mine titanium there, initially seeking permits over roughly 2,400 acres before narrowing its proposal and promising to mine only small sections at a time. Even so, opponents argued that any mining so close to the swamp posed unacceptable risks to one of the most ecologically significant wetlands in North America.

The Okefenokee is no ordinary wetland. Fed almost entirely by rainwater, it is the largest intact blackwater swamp on the continent. It serves as the headwaters of both the Suwannee and St. Marys rivers, stores enormous quantities of carbon in peat deposits up to 15 feet deep, and supports an extraordinary range of life, from black bears and red-cockaded woodpeckers to carnivorous plants and rare reptiles and amphibians.

According to Georgia’s Department of Natural Resources, the land it plans to acquire holds especially high conservation value. Species found there include the gopher tortoise, eastern indigo snake, gopher frog, Florida pine snake, Florida sandhill crane, and blackbanded sunfish.

Under state ownership, the primary goal will be to maintain high-quality wildlife habitat, while eventually opening the property to the public for activities such as hunting and fishing, potentially by 2027."""

In [11]:
# If needed:
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy
from collections import Counter
import math

nlp = spacy.load("en_core_web_sm")

STOPWORDS = nlp.Defaults.stop_words

def extractive_summary_spacy(text, max_sentences=3):
    """
    Extractive summary using spaCy sentence segmentation + word frequency scoring.
    Returns the top-ranked sentences in original order.
    """
    if not text or not text.strip():
        return ""

    doc = nlp(text.strip())
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    if len(sentences) <= max_sentences:
        return " ".join(sentences)

    # Build word frequency table
    words = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.lower_ not in STOPWORDS
            and len(token.text) > 2
        ):
            words.append(token.lower_)

    if not words:
        return " ".join(sentences[:max_sentences])

    freq = Counter(words)
    max_freq = max(freq.values())
    for word in freq:
        freq[word] /= max_freq

    # Score each sentence
    sentence_scores = {}
    for i, sent in enumerate(doc.sents):
        tokens = [
            token.lower_
            for token in sent
            if token.is_alpha and not token.is_stop and len(token.text) > 2
        ]

        if len(tokens) < 4:
            continue

        score = sum(freq.get(tok, 0) for tok in tokens)
        score /= math.log(len(tokens) + 1)  # mild length normalization
        sentence_scores[i] = score

    if not sentence_scores:
        return " ".join(sentences[:max_sentences])

    # Select top sentences, keep original order
    top_indices = sorted(
        sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:max_sentences]
    )

    summary = " ".join(sentences[i] for i in top_indices)
    return summary

In [14]:
print(extractive_summary(article, max_sentences=2))

After years of fierce resistance from conservationists, scientists, local residents, and lawmakers, thousands of acres of land near the Okefenokee National Wildlife Refuge will now be protected instead of turned into a titanium mine. The state of Georgia plans to acquire around 4,000 acres near the refuge from The Conservation Fund, the group that last year purchased nearly 8,000 acres of land and mineral rights from Twin Pines Minerals in order to stop the long-contested mining proposal.


## Summarizer by LORA method (distil-bart pretrained model)

In [2]:
## We are using XSum dataset: https://huggingface.co/datasets/EdinburghNLP/xsum/viewer/default/train

In [5]:
# ## Load XSum dataset
# from datasets import load_dataset

# # XSum has train/validation/test splits
# ds = load_dataset("xsum")

# # Use 10k training samples for a lightweight first run
# train_ds = ds["train"].select(range(10_000))

# # Keep a small validation slice
# val_ds = ds["validation"].select(range(500))

In [7]:
print(train_ds[:1])

{'document': ['The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.\nJeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.\nHowever, she said more preventative work could have been carried out to ensure the retaining wall did not fail.\n"It is difficult but I do think there is so much publicity for Dumfries and the Nith - and I totally appreci

In [8]:
## Add another prompt
PROMPT = (
    "Summarize the following news article in one short, factual paragraph. "
    "Keep it concise, neutral, and avoid bullet points.\n\nArticle:\n"
)

def add_prompt(example):
    return {
        "text": PROMPT + example["document"].strip(),
        "target": example["summary"].strip(),
    }

train_ds = train_ds.map(add_prompt, remove_columns=train_ds.column_names)
val_ds = val_ds.map(add_prompt, remove_columns=val_ds.column_names)

train_ds[0]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'text': 'Summarize the following news article in one short, factual paragraph. Keep it concise, neutral, and avoid bullet points.\n\nArticle:\nThe full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.\nJeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.\nHowever, she said more preventative work could have been carried out to ensure the retaining

In [9]:
### Tokenize 
from transformers import AutoTokenizer

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_length = 1024
max_target_length = 128

def tokenize_fn(example):
    model_inputs = tokenizer(
        example["text"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        text_target=example["target"],
        max_length=max_target_length,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(tokenize_fn, batched=False, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_fn, batched=False, remove_columns=val_ds.column_names)

train_tok[0].keys()

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [10]:
## Load pre-trained model and fine-tune with LoRA

import torch
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

trainable params: 786,432 || all params: 306,296,832 || trainable%: 0.2568


In [13]:
# ## Training for 3 epochs
# from transformers import (
#     DataCollatorForSeq2Seq,
#     Seq2SeqTrainingArguments,
#     Seq2SeqTrainer,
# )

# data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# training_args = Seq2SeqTrainingArguments(
#     output_dir="./distilbart-lora-xsum",
#     learning_rate=5e-5,
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=8,
#     num_train_epochs=3,
#     predict_with_generate=True,
#     generation_max_length=128,
#     save_total_limit=2,
#     logging_steps=50,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     fp16=torch.cuda.is_available(),
#     report_to="none",
# )

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_tok,
#     eval_dataset=val_tok,
#     processing_class=tokenizer,
#     data_collator=data_collator,
# )

# trainer.train()

In [ ]:
## Load XSum dataset
from datasets import load_dataset

# XSum has train/validation/test splits
ds = load_dataset("xsum")

# Use 10k training samples for a lightweight first run
train_ds = ds["train"].select(range(10_000))

# Keep a small validation slice
val_ds = ds["validation"].select(range(500))

## Add another prompt
PROMPT = (
    "Summarize the following news article in one short, factual paragraph. "
    "Keep it concise, neutral, and avoid bullet points.\n\nArticle:\n"
)

def add_prompt(example):
    return {
        "text": PROMPT + example["document"].strip(),
        "target": example["summary"].strip(),
    }

train_ds = train_ds.map(add_prompt, remove_columns=train_ds.column_names)
val_ds = val_ds.map(add_prompt, remove_columns=val_ds.column_names)

### Tokenize 
from transformers import AutoTokenizer

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_length = 1024
max_target_length = 128

def tokenize_fn(example):
    model_inputs = tokenizer(
        example["text"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        text_target=example["target"],
        max_length=max_target_length,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(tokenize_fn, batched=False, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_fn, batched=False, remove_columns=val_ds.column_names)

## Load pre-trained model and fine-tune with LoRA

import torch
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## Training for 3 epochs
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./distilbart-lora-xsum",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=128,
    save_total_limit=2,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

save_dir = "./distilbart-lora-news"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def generate_predictions(dataset, num_samples=50):
    preds = []
    refs = []

    sample_ds = dataset.select(range(min(num_samples, len(dataset))))

    for row in sample_ds:
        inputs = tokenizer(
            row["text"],
            return_tensors="pt",
            truncation=True,
            max_length=max_input_length,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                length_penalty=2.0,
                no_repeat_ngram_size=3,
            )

        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(row["target"])

    return preds, refs

preds, refs = generate_predictions(val_ds, num_samples=100)
results = rouge.compute(predictions=preds, references=refs)

print("######### RESULTS ##########")
print(results)



In [24]:
import zipfile
import os

zip_path = "C:/Users/abhin/Downloads/distilbart-lora-news2.zip"
extract_path = "./distilbart-lora-news2"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)

Extracted to: ./distilbart-lora-news2


In [25]:
import os
print(os.listdir("./distilbart-lora-news2"))

['adapter_config.json', 'adapter_model.safetensors', 'README.md', 'tokenizer.json', 'tokenizer_config.json']


In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

model_name = "sshleifer/distilbart-cnn-12-6"
lora_path = "./distilbart-lora-news2"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer from your folder
tokenizer = AutoTokenizer.from_pretrained(lora_path)

# Load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Attach your trained LoRA adapter
model = PeftModel.from_pretrained(base_model, lora_path)

model.to(device)
model.eval()

print("Model loaded successfully")

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model loaded successfully


In [27]:
PROMPT = (
    "Summarize the following news article in one short, factual paragraph. "
    "Keep it concise and neutral.\n\nArticle:\n"
)

def summarize(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [28]:
test_article = """
A new renewable energy initiative in California has expanded solar infrastructure
to thousands of homes, significantly reducing emissions while creating local jobs. This happened after a long standing public-private debate over the conervation efforts, originating from a 1994 protest.
"""

print("=== YOUR MODEL OUTPUT ===\n")
print(summarize(PROMPT + test_article))

=== YOUR MODEL OUTPUT ===



Both `max_new_tokens` (=64) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A new renewable energy initiative in California has expanded solar energy to thousands of homes in the state of the state, according to a report published by the U.S. state Department of Energy and the Department of Public Energy, it has been revealed that it is the result of a public-private effort to


In [29]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def summarize_base(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=64)

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("=== BASE MODEL ===\n")
print(summarize_base(PROMPT + test_article))

print("\n=== LORA MODEL ===\n")
print(summarize(PROMPT + test_article))

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

=== BASE MODEL ===



Both `max_new_tokens` (=64) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 A new renewable energy initiative in California has expanded solar infrastructure to thousands of homes, significantly reducing emissions while creating local jobs . This happened after a long standing public-private debate over the conervation efforts, originating from a 1994 protest . The initiative was created after a public-public debate started in 1994 .

=== LORA MODEL ===



Both `max_new_tokens` (=64) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A new renewable energy initiative in California has expanded solar energy to thousands of homes in the state of the state, according to a report published by the U.S. state Department of Energy and the Department of Public Energy, it has been revealed that it is the result of a public-private effort to
